
# Publicación en el `Online Feature Store`

**Autor**: Juan Carlos Alfaro Jiménez

El objetivo de esta libreta es mostrar cómo publicar las tablas de la capa `Gold` en un `Online Feature Store` externo (`Azure Cosmos DB`) para inferencia en tiempo real.

A diferencia de las instancias nativas de `Lakebase` (disponibles solo en niveles de pago), el uso de `Azure Cosmos DB` nos permite operar dentro de la **capa gratuita** de `Azure` manteniendo una integración profesional y de baja latencia.

Esta libreta **no es un *pipeline* de _streaming_**. Es un *script* de configuración que:

* **No transforma datos**: Sincroniza las tablas `Delta` ya existentes en `Unity Catalog` hacia el almacén en línea.
* **Usa claves primarias**: El `Feature Store` reconoce las tablas gracias a los `CONSTRAINT` de clave primaria definidos previamente.
* **Habilita el servicio**: Permite que el modelo recupere características en milisegundos durante la inferencia.

La arquitectura que publicaríamos es la siguiente:

* **`gold_fraud_spine`**: esqueleto de entrenamiento. **No se publica** aquí porque no es una tabla de características: es el andamio que el *job* de entrenamiento usa para buscar las características. Nunca se consulta durante la inferencia.
* **`gold_customer_profile`**: reconocida automáticamente como tabla de características por su clave primaria `(customer_id, __START_AT TIMESERIES)`. Se publicaría en el `Online Store` para que la capa de *serving* recupere el perfil demográfico actual del cliente en tiempo real.
* **`gold_customer_aggregations`**: reconocida automáticamente como tabla de características por su clave primaria `(customer_id, window_end TIMESERIES)`. Se publicaría en el `Online Store` para consultas de señales de comportamiento durante la inferencia.

> **Limitación de la capa gratuita**: aunque `Databricks` ofrece una solución nativa basada en `Lakebase`, esta incurre en costes constantes. En este proyecto, utilizamos la integración con `Azure Cosmos DB` para garantizar un entorno de aprendizaje a coste cero, aprovechando los créditos de `Azure for Students+  y el `Free Tier`.

**¿Cuándo volver a ejecutar esta libreta?**

* Configuración inicial de la conexión entre `Databricks` y `Azure`.
* Cuando se recrea la infraestructura de `Azure Cosmos DB` o se añaden nuevas tablas de características.


## 1. Importación de librerías y configuración

Cargamos el cliente del `Feature Store` (`FeatureEngineeringClient`) y definimos los nombres completamente cualificados de las tablas sobre las que vamos a operar. En `Unity Catalog`, un nombre completamente cualificado sigue el formato `catálogo.esquema.tabla`.

A diferencia de la lectura de tablas `Delta`, la publicación en el `Online Feature Store` requiere definir una especificación de conexión que apunte a nuestra instancia de `Azure Cosmos DB` de forma segura, utilizando el gestor de secretos de `Databricks` para proteger las claves de acceso.

In [0]:
#
from databricks.feature_engineering import FeatureEngineeringClient

#
from databricks.feature_engineering.online_store_spec import AzureCosmosDBSpec

In [0]:
# Fully qualified table names in Unity Catalog (catalog.schema.table)
catalog  = "workspace"
database = "credit_card_fraud"

gold_customer_profile_table = f"{catalog}.{database}.gold_customer_profile"
gold_customer_aggregations_table = f"{catalog}.{database}.gold_customer_aggregations"

# A single online store can host multiple feature tables.
# This is the recommended approach to reduce infrastructure costs.
online_store_name = "credit_card_fraud_online_store"

# Names for the tables once published inside the online store
online_profile_table = "online_customer_profile"
online_aggregations_table = "online_customer_aggregations"

#
account_uri = "https://cosmos-fraud-feature-store.documents.azure.com:443/"  #
read_secret_prefix = "azure-scope/cosmos"  #
write_secret_prefix = "azure-scope/cosmos"  #

online_store_spec = AzureCosmosDBSpec(
  account_uri = account_uri,
  database_name = database,
  read_secret_prefix = read_secret_prefix,
  write_secret_prefix = write_secret_prefix
)

# Instantiate the client using the current cluster credentials automatically
fe = FeatureEngineeringClient()


## 2. Configuración del `Online Feature Store` (`Azure Cosmos DB`)

En este proyecto, el `Online Feature Store` está respaldado por `Azure Cosmos DB` una base de datos `NoSQL` de alto rendimiento que permite servir características con **baja latencia** para inferencia en tiempo real. Esta pieza es fundamental para que el modelo de serving recupere el perfil y el comportamiento del cliente de forma inmediata, evitando las latencias de lectura de las tablas Delta originales.

Al utilizar un **almacén de terceros**, la gestión de la capacidad y el rendimiento  se realiza directamente desde el portal de `Azure`, lo que nos permite integrarnos con la **capa gratuita** y evitar las restricciones de licencia de `Lakebase` en entornos académicos.

El flujo de datos sincronizado funciona de la siguiente manera:

1. El pipeline `Gold` escribe nuevas agregaciones en las tablas `Delta` de `Unity Catalog`.
2. La función `publish_table` sincroniza estas tablas con `Azure Cosmos DB` utilizando el modo `merge` para actualizaciones eficientes.
3. Durante la inferencia, el modelo consulta el `Online Feature Store` por `customer_id` y obtiene las características necesarias en milisegundos.
4. El modelo predice el riesgo de fraude y devuelve el resultado en tiempo real.

> Nota: A diferencia de `Lakebase`, no es necesario ejecutar un comando de creación de instancia (create_online_store) desde Databricks. Simplemente utilizamos el objeto online_store_spec definido previamente para indicar a Databricks dónde debe "empujar" los datos.

In [0]:
#
print(f"Online store target: {online_store_spec.account_uri}")
print(f"Secret scope prefix: {online_store_spec.read_secret_prefix}")

#
try:
    # We check if the spec is correctly formed before proceeding to the publication step
    if online_store_spec.account_uri and online_store_spec.read_secret_prefix:
        print("SUCCESS: Online store specification is valid and ready for publishing.")
    else:
        print("ERROR: Please check the account_uri and secret prefixes in Section 1.")
except Exception as e:
    print(f"Configuration error: {e}")


## 3. Publicación de tablas de características

El paso final es la **publicación**. Este proceso toma los datos de la capa `Gold` (almacenados en formato `Delta`) y los sincroniza con el contenedor de `Azure Cosmos DB`.

Utilizamos el modo `merge`, que es el único soportado para almacenes de terceros, para asegurar que solo se actualicen los registros que han cambiado, optimizando así el uso de nuestra capa gratuita. Una vez ejecutado este paso, las características estarán disponibles para ser consultadas por el modelo de *serving* con una latencia de milisegundos.

In [0]:
# Specific specification for the Customer Profile table
profile_store_spec = AzureCosmosDBSpec(
  account_uri = account_uri,
  database_name = database,             # "credit_card_fraud"
  container_name = online_profile_table, # "online_customer_profile"
  read_secret_prefix = read_secret_prefix,
  write_secret_prefix = write_secret_prefix
)

# Specific specification for the Aggregations table
aggregations_store_spec = AzureCosmosDBSpec(
  account_uri = account_uri,
  database_name = database,                  # "credit_card_fraud"
  container_name = online_aggregations_table, # "online_customer_aggregations"
  read_secret_prefix = read_secret_prefix,
  write_secret_prefix = write_secret_prefix
)

In [0]:
mode = "merge"  #

#
fe.publish_table(
  name = gold_customer_profile_table,
  online_store = profile_store_spec,
  mode = mode
)

#
fe.publish_table(
    name = gold_customer_aggregations_table,
    online_store = aggregations_store_spec,
    mode = mode
)


## 4. Conclusiones y siguientes pasos

### ¿Qué hemos visto?

En esta libreta hemos descrito la arquitectura de publicación en el `Online Store`:

1. Las tablas `Delta` de la capa `Gold` son reconocidas automáticamente por el `Feature Store` gracias al `CONSTRAINT` de clave primaria declarado en el pipeline.
2. En un entorno con licencia completa, `fe.create_online_store` crearía una instancia `PostgreSQL` gestionada por `Lakebase` con latencia de *serving* inferior a 10 ms.
3. `fe.publish_table` en modo `CONTINUOUS` establecería un canal de sincronización permanente entre las tablas `Delta` y el `Online Store`, de modo que el modelo siempre consultaría las características más recientes.
4. Durante la inferencia, el modelo recuperaría las características automáticamente a través de la `API` del `Feature Store`, sin necesidad de hacer `JOIN` explícitos en el código.

### ¿Qué sigue?

En la siguiente libreta construiremos el conjunto de datos de entrenamiento mediante la `API` `create_training_set`, que realiza automáticamente los joins `PiT` entre la *spine* (`gold_fraud_spine`) y las dos tablas de características
(`gold_customer_profile` y `gold_customer_aggregations`) leyendo directamente desde las tablas `Delta` de la capa `Gold`. El `Online Store` no interviene en el entrenamiento: su único propósito es servir características con baja latencia durante la inferencia en tiempo real.